# Module 2 — Advanced Cleaning

**By the end of this notebook, you will be able to:**
- Explain why a per-city cleaning strategy that works on two well-populated cities can silently fail on a much smaller one
- Decide when a column should be dropped rather than filled, based on how much of it is actually missing
- Apply that decision *before* filling, and know why the order matters

**Context:** Session 2 worked with two cities picked to make a manual train/test split easy to reason about: Kampala (5596 rows) and Nairobi (1500 rows), capped at 1200 rows each. The full dataset has two more cities — Lagos (852 rows) and Bujumbura (only 123 rows) — and nothing capped. This module works with all four, uncapped, and finds out why `fill_missing_by_city` alone is not enough once a city gets this small.

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

from air_quality import data, evaluation, features
from air_quality.workflows import DEFAULT_COLUMNS

ALL_CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]

train_df, _ = data.load_datasets()
train_df["city"].value_counts()

## The real imbalance

Session 2 used two cities specifically because they are both large enough for a manual split to be readable. `value_counts()` above shows what the other two look like: Lagos has roughly a third of Nairobi's rows, and Bujumbura has barely more than a hundred — 45x fewer than Kampala. Any per-city operation, like filling missing values city by city, has 45x less data to work with there.

**Exercise:** Call `data.restrict_to_scope(train_df, ALL_CITIES, DEFAULT_COLUMNS)` — no `max_rows_per_city` this time, since you want the real imbalance, not a capped-down version of it.

In [ ]:
# TODO 1: scope train_df to ALL_CITIES and DEFAULT_COLUMNS, without
# capping rows per city


## Missing values, by city this time

Session 2's discovery notebook already showed you a per-city missing-value heatmap, on two cities. Reapply the exact same pattern here, on `scoped` — only the number of cities changed.

In [ ]:
# TODO 2: percentage of missing values per column, broken down by city
# — same pattern as the discovery notebook's missing-values heatmap


**Look at `uvaerosollayerheight_aerosol_height`.** It was already the most-missing column in Session 2 (93-97% missing in Kampala/Nairobi) — high, but `fill_missing_by_city`'s forward/backward-fill could still work with the few values that did exist in each city. What does the heatmap show for Bujumbura?

## Why forward/backward-fill breaks down

`fill_missing_by_city` (Session 2, `data.py`) fills each city using only that city's own values, sorted by date — it never borrows a value from another city. If a city has **zero** non-missing values for a column, there is nothing to forward- or backward-fill from: the column stays entirely missing for that city's rows.

Bujumbura is missing `uvaerosollayerheight_aerosol_height` **100% of the time**. Apply the fill exactly as Session 2 did and see what remains.

In [ ]:
# Given: apply the Session 2 fill as-is, across all four cities this time
fillable_columns = [c for c in scoped.columns if c not in ("city", "date")]
naively_filled = data.fill_missing_by_city(scoped, fillable_columns)
naively_filled.isna().sum()[naively_filled.isna().sum() > 0]

123 rows — every single Bujumbura row — still have a missing value in that column. Nothing stops it from silently reaching the model. See what that actually does: train on Kampala, evaluate on Bujumbura, using this naively-filled data.

In [ ]:
# Given: this is expected to fail — the point is to see the exact error
enriched = features.add_temporal_features(naively_filled)
train_split = enriched[enriched["city"] == "Kampala"]
test_split = enriched[enriched["city"] == "Bujumbura"]
feature_cols = features.feature_columns(enriched)

try:
    evaluation.evaluate_manual_split(LinearRegression(), train_split, test_split, feature_cols)
except ValueError as error:
    print("Training on Kampala, evaluating on Bujumbura crashed:")
    print(error)

## The fix: drop before filling

The problem is not the filling strategy — it is trying to fill a column that has no data at all in a given city. The right move is to drop columns that are missing **too much data overall** *before* filling the rest per city, not after.

Open `src/air_quality/data.py` and complete the two functions in the "Session 3 — optional modules" section at the bottom of the file: `columns_above_missing_threshold` and `drop_columns`. Their docstrings specify the exact contract, and `tests/test_data_advanced.py` specifies the exact expected behavior — read it the same way you read `tests/test_data.py` in Session 2's Transfer to Python activity. Run `uv run pytest tests/test_data_advanced.py -v` until both pass, then come back here.

**Exercise:** Use `data.columns_above_missing_threshold(scoped, threshold=0.7)` to find the columns to drop, then `data.drop_columns` to remove them from `scoped`. Fill what remains with `data.fill_missing_by_city`, and confirm no missing values remain.

In [ ]:
# TODO 3: drop columns missing more than 70% of the time, then fill the
# rest per city, and confirm nothing is missing anymore


**Question:** Why does dropping have to happen *before* filling, not after? What would `columns_above_missing_threshold` report if you called it on `naively_filled` from earlier instead of on `scoped`? And why 0.7 specifically — what would change with a lower threshold, like 0.5?

## Confirming the fix

Rerun the same Kampala → Bujumbura evaluation as before, this time on `cleaned` instead of `naively_filled`.

In [ ]:
# Given: same evaluation as before, on the cleaned data this time
enriched = features.add_temporal_features(cleaned)
train_split = enriched[enriched["city"] == "Kampala"]
test_split = enriched[enriched["city"] == "Bujumbura"]
feature_cols = features.feature_columns(enriched)

metrics = evaluation.evaluate_manual_split(LinearRegression(), train_split, test_split, feature_cols)
metrics

## From notebook to pipeline

This entire sequence — scope to all four cities, drop high-missing columns, fill the rest per city, engineer features, evaluate a manual split — is what `run_advanced` (in `src/air_quality/workflows.py`) should become. Unlike `run_baseline`, it is not given to you complete: you write and maintain it yourself, module by module, starting here. Open `workflows.py`, find `AdvancedPipelineConfig` and the `run_advanced` stub, and write its body — reuse exactly what you did above (scope, drop, fill, add temporal features, then `evaluate_manual_split` with `config.train_city`/`config.test_city`). There is no test for this one: verify it by calling it below and checking the metrics match what you got above.

In [ ]:
# Once you have written run_advanced in workflows.py, this should print
# the same metrics you got above
from air_quality.workflows import run_advanced

run_advanced()

## Wrap-up

Write down: which column got dropped, at what threshold, and in one sentence, why a per-city fill needs a missingness check first once your cities are this imbalanced.

_Your observations here._